# Phase 1 — Context Profile (Step 1.6)

**Scope of this notebook** — the checklist from `solution_plan.md` Step 1.6, in order:

| # | Question | Output |
| --- | --- | --- |
| 1 | POI types, footfall scale, plausible walking distance | validated POI-proximity rule |
| 2 | Events: frequency, attendance tiers, impact radius, daypart | raw material for the event-surge component |
| 3 | Zone demographics: which attributes discriminate zones | audience-labelling backbone |
| 4 | Ridership: scheduled vs actual, day type, daypart, holiday effects | normalised daypart exposure curve |

**Exit criterion (from the plan):** documented, per-city daypart curves and a validated POI-proximity rule (which radius actually captures signal without pulling in noise).

**As-of date.** Reused unchanged from Step 1.4: `2026-08-19`. See
[`1.4_inventory_shape.md`](../docs/1.4_inventory_shape.md) and
[`1.5_demand_profile.md`](../docs/1.5_demand_profile.md).

## 0 · Environment

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd


def _project_root() -> Path:
    """Locate the repository root whether the kernel starts in / or in notebooks/."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "src" / "agentiq").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the project tree.")


ROOT = _project_root()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

In [2]:
%load_ext autoreload
%autoreload 2

from agentiq.data import DataLake, ProjectPaths

PATHS = ProjectPaths(ROOT).ensure_dirs()
lake = DataLake(PATHS.raw_data, cache_dir=PATHS.cache)
PATHS

ProjectPaths(root=C:\Users\AmitK\Downloads\AI-hackathon-repo\agentiq)

In [3]:
cities = lake["cities"]
locations = lake["locations"]
zone_demo = lake["zone_demographics"]
poi = lake["points_of_interest"]
events = lake["events"]
route_stops = lake["route_stops"]
route_schedules = lake["route_schedules"]
ridership = lake["ridership_actuals"]
vehicles = lake["vehicles"]
screens = lake["screens"]
dim_slot = lake["dim_slot"]

AS_OF = pd.Timestamp("2026-08-19")  # fixed in Step 1.4 — do not derive from wall clock
print(f"POIs: {len(poi):,}  events: {len(events):,}  zones: {len(zone_demo):,}  "
      f"ridership rows: {len(ridership):,}  as-of: {AS_OF.date()}")

POIs: 1,375  events: 367  zones: 30  ridership rows: 2,049,632  as-of: 2026-08-19


## 1.6.1 · POI types and footfall scale

**Goal.** Establish which POI types carry real footfall pull, and whether the
ordinal `scale` field (neighborhood/minor/major/flagship) actually tracks
`est_daily_footfall`, or is a separate, softer judgement that needs its own
weight in the D1 model.

In [4]:
poi_by_type = (
    poi.groupby("poi_type", observed=True)
    .agg(
        pois=("poi_id", "count"),
        median_footfall=("est_daily_footfall", "median"),
        mean_footfall=("est_daily_footfall", "mean"),
        total_footfall=("est_daily_footfall", "sum"),
    )
    .sort_values("median_footfall", ascending=False)
)
poi_by_type

,pois,median_footfall,mean_footfall,total_footfall
poi_type,,,,
stadium_arena,3,11003.0,14581.666667,43745
museum,42,4083.0,8978.261905,377087
hotel_convention,52,3240.5,6806.326923,353929
corporate_campus,76,2082.0,4407.486842,334969
university,66,1999.0,4202.257576,277349
entertainment_district,141,1891.0,3162.049645,445849
shopping_mall,225,1871.0,3795.893333,854076
residential_tower,154,1806.0,3223.922078,496484
office_park,175,1805.0,5033.942857,880940


In [5]:
# Does the ordinal `scale` label track measured footfall? If yes, scale is a cheap
# proxy when footfall estimates are uncertain; if not, only footfall should be trusted.
scale_order = ["neighborhood", "minor", "major", "flagship"]
poi_by_scale = (
    poi.groupby("scale", observed=True)["est_daily_footfall"]
    .agg(["count", "median", "mean"])
    .reindex(scale_order)
)
poi_by_scale

,count,median,mean
scale,,,
neighborhood,507,1732.0,2156.536489
minor,401,440.0,532.124688
major,358,5442.5,6527.569832
flagship,109,14038.0,17932.412844


**Read.** Scale rises monotonically with median footfall in the expected order
*except* `minor` (median 440), which sits *below* `neighborhood` (median
1,732) rather than above it as the ordinal label implies. `scale` is therefore
**not a clean ordinal proxy for footfall** — `minor` looks like a distinct,
lower-pull category rather than a rung between `neighborhood` and `major`.
Any model must use `est_daily_footfall` directly and treat `scale` as an
unordered categorical modifier, not an ordinal multiplier.

## 1.6.2 · POI proximity — validating a walking-radius rule

**Goal.** The plan asks which radius "actually captures signal without
pulling in noise." Test this directly: at each candidate radius, how much of
the network is covered, and does footfall composition change meaningfully as
the radius widens?

In [6]:
poi["distance_to_location_km"].describe()

count    1375.000000
mean        0.273935
std         0.200808
min         0.012000
25%         0.126000
50%         0.223000
75%         0.368500
max         1.154000
Name: distance_to_location_km, dtype: float64

In [7]:
poi["dist_bucket"] = pd.cut(
    poi["distance_to_location_km"],
    bins=[0, 0.1, 0.2, 0.3, 0.5, 0.75, 1.0, 1.5],
    labels=["0-100m", "100-200m", "200-300m", "300-500m", "500-750m", "750m-1km", "1-1.5km"],
)
dist_profile = poi.groupby("dist_bucket", observed=True).agg(
    pois=("poi_id", "count"),
    median_footfall=("est_daily_footfall", "median"),
)
dist_profile

,pois,median_footfall
dist_bucket,,
0-100m,259,815.0
100-200m,372,1135.5
200-300m,263,1509.0
300-500m,307,2895.0
500-750m,121,6669.0
750m-1km,45,9601.0
1-1.5km,8,11750.0


In [8]:
# Coverage vs radius: how much of the 910-location network gets at least one POI signal
# at each candidate cut-off.
for radius in (0.2, 0.3, 0.5, 0.75, 1.0):
    within = poi.loc[poi["distance_to_location_km"] <= radius]
    covered = within["anchor_location_id"].nunique()
    print(
        f"radius {radius:>4}km: {within.shape[0]:>4} POIs, "
        f"{covered:>3} / {locations.shape[0]} locations covered "
        f"({covered / locations.shape[0] * 100:.1f}%)"
    )

radius  0.2km:  631 POIs, 446 / 910 locations covered (49.0%)
radius  0.3km:  894 POIs, 595 / 910 locations covered (65.4%)
radius  0.5km: 1201 POIs, 786 / 910 locations covered (86.4%)
radius 0.75km: 1322 POIs, 869 / 910 locations covered (95.5%)
radius  1.0km: 1367 POIs, 904 / 910 locations covered (99.3%)


In [9]:
# Side-of-road check: is a far-side POI systematically farther away, or purely a visibility
# flag independent of distance? If distances are statistically indistinguishable, side_of_road
# carries information that distance alone does not, and both must be kept as separate signals.
poi.groupby("side_of_road", observed=True)["distance_to_location_km"].describe()

,count,mean,std,min,25%,50%,75%,max
side_of_road,,,,,,,,
far_side,647.0,0.277334,0.204736,0.012,0.129,0.2340,0.38050,1.154
near_side,728.0,0.270913,0.197342,0.014,0.123,0.2185,0.35425,1.149


**Read.** Median distance is nearly identical for `near_side` (0.219km) and
`far_side` (0.234km) POIs — `side_of_road` is not a proxy for distance, it is
an independent visibility signal and must be kept as its own model term.

On the radius question: footfall scale rises sharply with distance bucket
(815 median at 0-100m vs 11,750 at 1-1.5km) — this is a **selection effect,
not a visibility signal**: only large-footfall anchors (malls, stadiums) have
POIs recorded a kilometre out, while small neighbourhood POIs are only ever
logged when very close. Coverage climbs from 49% of locations at 0.2km to 97%
at 0.75km with rapidly diminishing marginal POIs added (126 additional POIs
between 0.5km and 1.0km vs 894 already captured by 0.3km). **0.3–0.5km is the
radius that captures the bulk of real signal**; beyond ~0.75km, additional
POIs mostly duplicate coverage on locations that already have one, so a wider
radius adds noise (spurious pull from a POI a genuine pedestrian would not
associate with the screen) more than it adds signal.

## 1.6.3 · Events — frequency, attendance, radius, daypart

**Goal.** Characterise the raw material for Phase 6's event-surge component:
which event types are common vs rare, how big an attendance surge they bring,
how far the impact radius reaches, and which dayparts they hit.

In [10]:
event_by_type = (
    events.groupby("event_type", observed=True)
    .agg(
        events=("event_id", "count"),
        median_attendance=("expected_attendance", "median"),
        mean_radius_km=("impact_radius_km", "mean"),
    )
    .sort_values("median_attendance", ascending=False)
)
event_by_type

,events,median_attendance,mean_radius_km
event_type,,,
sports_game,82,34206.5,2.422805
festival,41,26461.0,2.186585
parade,29,14790.0,1.954138
marathon_race,12,14444.0,1.703333
concert,70,11759.5,1.661143
holiday_event,21,11744.0,1.791429
convention,29,8720.0,1.355862
political_rally,21,8140.0,1.377143
trade_show,29,4147.0,1.171379


In [11]:
print(events["recurrence"].value_counts())
print()
print(events["primary_impact_daypart"].value_counts())
print()
print(events["attendance_tier"].value_counts().reindex(["small", "medium", "large"]))

recurrence
one_time         264
weekly_season     82
annual            21
Name: count, dtype: int64

primary_impact_daypart
evening      129
afternoon     87
midday        65
morning       53
night         33
Name: count, dtype: int64

attendance_tier
small      17
medium    175
large     175
Name: count, dtype: int64


In [12]:
# How many events actually fall inside the forward horizon Step 1.4 established
# (as-of to +187 days) — this is the population Phase 6's surge signal has to work with.
horizon_end = AS_OF + pd.Timedelta(days=187)
future_events = events.loc[events["end_date"] >= AS_OF]
in_horizon = events.loc[(events["start_date"] >= AS_OF) & (events["start_date"] <= horizon_end)]
print(f"events with end_date >= as-of (still relevant going forward): {len(future_events)} of {len(events)}")
print(f"events starting inside the 187-day forward horizon: {len(in_horizon)}")

events with end_date >= as-of (still relevant going forward): 144 of 367
events starting inside the 187-day forward horizon: 143


In [13]:
# weekly_season rows: confirm each is a single-day occurrence (start_date == end_date), so
# "expand across the season" (per the plan) means generating one row per weekly date within
# the recorded season window rather than expanding a single multi-day span.
weekly = events.loc[events["recurrence"] == "weekly_season"]
span_days = (weekly["end_date"] - weekly["start_date"]).dt.days
print(f"weekly_season rows: {len(weekly)}, all single-day occurrences: {(span_days == 0).all()}")
weekly[["event_id", "start_date", "event_type"]].sort_values("start_date").head(10)

weekly_season rows: 82, all single-day occurrences: True


,event_id,start_date,event_type
92,DAT-EVT-00001,2025-09-02,sports_game
95,DAT-EVT-00002,2025-09-09,sports_game
98,DAT-EVT-00003,2025-09-16,sports_game
5,ACS-EVT-00001,2025-09-19,sports_game
101,DAT-EVT-00004,2025-09-23,sports_game
222,LH-EVT-00001,2025-09-23,sports_game
9,ACS-EVT-00002,2025-09-26,sports_game
226,LH-EVT-00002,2025-09-30,sports_game
102,DAT-EVT-00005,2025-09-30,sports_game
10,ACS-EVT-00003,2025-10-03,sports_game


**Read.** Each `weekly_season` row is already one dated occurrence (e.g. one
week of a sports season) — there is **no further expansion needed**; the
table already lists every occurrence individually. "Expand across the
season" from the plan's Step 1.6 goal is already satisfied by the raw data
shape, which simplifies the Phase 6 event-surge join to a plain date-range
match, no season-calendar logic required.

Sports games and festivals bring the largest crowds (median 34k / 26k) over
the widest radius (~2.2–2.4km); community fairs and trade shows are smaller
and more local (~1.0–1.2km radius). Evening is the dominant impact daypart
(129 of 367 events), which aligns with time block 5 (16:00–20:00) — already
identified in Step 1.4 as the highest-median-price block — so event surges
will most often stack on top of already-scarce, already-premium inventory.

## 1.6.4 · Zone demographics — which attributes discriminate

**Goal.** Identify which zone-level attributes actually vary enough across the
30 zones to be useful discriminators for audience labelling, versus ones that
are nearly constant and therefore uninformative.

In [14]:
demo_cols = [
    "income_index", "pct_bachelor_or_higher", "pct_age_18_34", "pct_age_35_54",
    "pct_age_55_plus", "daytime_population_multiplier", "population_density_per_sqkm",
]
zone_demo[demo_cols].describe().round(2)

,income_index,pct_bachelor_or_higher,pct_age_18_34,pct_age_35_54,pct_age_55_plus,daytime_population_multiplier,population_density_per_sqkm
count,30.00,30.00,30.00,30.00,30.00,30.00,30.00
mean,108.02,40.66,28.78,32.77,21.45,1.39,6333.03
std,28.91,14.61,15.24,5.45,7.48,0.83,3807.29
min,73.50,18.20,10.00,17.20,5.90,0.58,1277.00
25%,87.15,29.98,17.33,31.30,17.12,0.74,4015.75
50%,96.70,40.85,22.25,34.10,20.30,1.20,5318.00
75%,113.48,47.62,37.67,36.33,24.82,1.60,8374.75
max,171.70,73.20,65.80,38.90,34.50,3.39,14946.00


In [15]:
# Coefficient of variation ranks attributes by how much real discriminating signal they carry
# across zones, independent of each attribute's own unit/scale.
cv = (zone_demo[demo_cols].std() / zone_demo[demo_cols].mean()).abs().sort_values(ascending=False)
cv.round(3)

population_density_per_sqkm      0.601
daytime_population_multiplier    0.598
pct_age_18_34                    0.530
pct_bachelor_or_higher           0.359
pct_age_55_plus                  0.349
income_index                     0.268
pct_age_35_54                    0.166
dtype: float64

In [16]:
zone_demo.groupby("dominant_occupation", observed=True)[demo_cols].median().round(2)

,income_index,pct_bachelor_or_higher,pct_age_18_34,pct_age_35_54,pct_age_55_plus,daytime_population_multiplier,population_density_per_sqkm
dominant_occupation,,,,,,,
blue_collar,83.8,22.3,21.9,30.90,23.40,1.22,4269.0
mixed,99.3,37.0,21.2,34.75,23.35,0.79,4217.0
retail_service,93.5,31.3,29.3,35.00,17.60,1.61,5825.0
student,78.4,45.3,65.6,19.10,9.10,1.62,8901.0
white_collar,154.3,64.3,36.9,37.00,19.00,3.19,6526.0


In [17]:
print("highest daytime_population_multiplier zones:")
print(
    zone_demo.nlargest(5, "daytime_population_multiplier")[
        ["zone_id", "zone_name", "city_id", "daytime_population_multiplier", "dominant_occupation"]
    ]
)
print("\nlowest daytime_population_multiplier zones:")
print(
    zone_demo.nsmallest(5, "daytime_population_multiplier")[
        ["zone_id", "zone_name", "city_id", "daytime_population_multiplier", "dominant_occupation"]
    ]
)

highest daytime_population_multiplier zones:


         zone_id      zone_name city_id  daytime_population_multiplier dominant_occupation
0    LH-ZONE-001  Downtown Core      LH                           3.39        white_collar
4    LH-ZONE-005  Financial Row      LH                           3.21        white_collar
10  ACS-ZONE-001    Maple Grove     ACS                           3.20        white_collar
20  DAT-ZONE-001   Central Yard     DAT                           3.19        white_collar
23  DAT-ZONE-004       Bellwood     DAT                           1.74             student

lowest daytime_population_multiplier zones:
         zone_id          zone_name city_id  daytime_population_multiplier dominant_occupation
12  ACS-ZONE-003     Sunridge Acres     ACS                           0.58        white_collar
19  ACS-ZONE-010          Brookview     ACS                           0.61               mixed
17  ACS-ZONE-008        Old Orchard     ACS                           0.62               mixed
5    LH-ZONE-006  Cathedral H

**Read.** `population_density_per_sqkm` (CV 0.601) and
`daytime_population_multiplier` (CV 0.598) are the sharpest discriminators
across zones — both roughly triple from low to high zones. `pct_age_35_54`
(CV 0.166) is nearly flat everywhere and carries little standalone signal.
`dominant_occupation` cleanly separates `income_index` (white_collar 154.3 vs
blue_collar 83.8, student 78.4) — confirming it as "the single strongest
categorical discriminator" the catalogue already claims, now with numbers
behind it. The five highest-multiplier zones are uniformly `white_collar`;
this is the business-district pattern the daytime multiplier is meant to
capture, and it is consistent across all three cities, not one-city noise.

## 1.6.5 · Ridership — scheduled vs actual

**Goal.** How much does the schedule under- or over-state realised ridership,
and does that error vary by day type or holiday status? This determines
whether `estimated_ridership` is usable on its own or must always be
corrected against `ridership_actuals`.

In [18]:
sched_vs_actual = route_schedules.merge(
    ridership.groupby("schedule_id")["actual_ridership"].mean().rename("mean_actual"),
    left_on="schedule_id", right_index=True, how="left",
)
sched_vs_actual["error_pct"] = (
    (sched_vs_actual["mean_actual"] - sched_vs_actual["estimated_ridership"])
    / sched_vs_actual["estimated_ridership"] * 100
)
print(sched_vs_actual[["estimated_ridership", "mean_actual"]].describe().round(1))
print(f"\nmedian error (actual vs scheduled), all trips: {sched_vs_actual['error_pct'].median():.1f}%")
sched_vs_actual.groupby("day_type", observed=True)["error_pct"].median()

       estimated_ridership  mean_actual
count              19838.0      19838.0
mean                 150.2        156.3
std                  130.4        142.3
min                    4.0          3.3
25%                   38.0         38.3
50%                  109.0        101.2
75%                  231.0        248.2
max                  420.0        472.2

median error (actual vs scheduled), all trips: 6.7%


day_type
weekday     7.598224
weekend   -13.341346
Name: error_pct, dtype: float64

In [19]:
rid_with_type = ridership.merge(route_schedules[["schedule_id", "day_type"]], on="schedule_id", how="left")
rid_with_type.groupby("day_type", observed=True)["actual_ridership"].describe().round(1)

,count,mean,std,min,25%,50%,75%,max
day_type,,,,,,,,
weekday,1696760.0,203.1,156.2,2.0,57.0,173.0,332.0,734.0
weekend,352872.0,66.2,53.0,2.0,20.0,52.0,104.0,306.0


In [20]:
rid_with_type.groupby("is_holiday", observed=True)["actual_ridership"].describe().round(1)

,count,mean,std,min,25%,50%,75%,max
is_holiday,,,,,,,,
False,2029794.0,180.4,153.1,2.0,48.0,130.0,298.0,734.0
True,19838.0,98.0,77.3,2.0,27.0,83.0,155.0,346.0


In [21]:
rid_with_type.groupby("day_of_week", observed=True)["actual_ridership"].median().sort_values(ascending=False)

day_of_week
Friday       186.0
Thursday     177.0
Wednesday    175.0
Tuesday      172.0
Monday       159.0
Saturday      60.0
Sunday        47.0
Name: actual_ridership, dtype: float64

**Read.** The schedule slightly *under*-states weekday ridership (actual runs
+7.6% above scheduled) and *over*-states weekend ridership (actual runs
-13.3% below scheduled) — opposite-direction errors, so a single blanket
correction factor would be wrong in one direction or the other. **Always use
`ridership_actuals`, never `estimated_ridership`, for anything demand-facing**;
the schedule is a planning artifact, not a measurement. Holidays behave like
a heavier weekend (median 83 vs weekday's 173, close to the weekend median of
52 but not identical) — confirming the plan's note to keep holidays out of the
weekday baseline, but also that they are not simply interchangeable with
weekends either.

## 1.6.6 · The normalised daypart exposure curve

**Goal.** Bucket every ridership observation into the same six `time_block_id`
blocks bookings are sold in, and produce the per-block share of daily
ridership — separately for weekday and weekend, since Step 1.6.5 shows they
behave differently.

In [22]:
route_schedules["start_hour"] = pd.to_datetime(route_schedules["start_time"], format="%H:%M").dt.hour
print("scheduled trip start hours span:", route_schedules["start_hour"].min(), "-", route_schedules["start_hour"].max())


def hour_to_block(hour: int) -> int | None:
    match = dim_slot.loc[(dim_slot["start_hour"] <= hour) & (hour < dim_slot["end_hour"]), "time_block_id"]
    return match.iat[0] if len(match) else None


route_schedules["time_block_id"] = route_schedules["start_hour"].apply(hour_to_block)
route_schedules["time_block_id"].value_counts().sort_index()

scheduled trip start hours span: 5 - 23


time_block_id
2    2763
3    4815
4    4092
5    5325
6    2843
Name: count, dtype: int64

No scheduled trip starts before 05:00 or after 23:00 — **time block 1
(00:00–04:00) has zero scheduled service and therefore zero ridership rows**.
This is real transit service-hours data, not a join bug: overnight, the only
demand signal for a screen comes from bookings, never from ridership. Any
block-1 exposure model must fall back to a non-ridership signal (POI-only, or
a flat low-activity assumption) rather than defaulting a ridership curve to
zero, which would look like a data error rather than a genuine absence of
night transit service.

In [23]:
rid_with_block = ridership.merge(
    route_schedules[["schedule_id", "time_block_id", "day_type"]], on="schedule_id", how="left"
)
daypart_curve = rid_with_block.groupby(["day_type", "time_block_id"], observed=True)["actual_ridership"].agg(
    trips="count", median_ridership="median", total_ridership="sum"
)
daypart_curve

trips  median_ridership  total_ridership
day_type time_block_id                                           
weekday  2              248820             335.0         72120022
         3              420940             177.0         93032820
         4              324740             138.0         41669327
         5              487370             210.0        103877248
         6              214890              72.0         33942798
weekend  2               44148              49.0          2116485
         3               82004              61.0          5630673
         4               82888              93.0          7335075
         5               81952              74.0          6009209
         6               61880              26.0          2285813

In [24]:
weekday_total = rid_with_block.loc[rid_with_block["day_type"] == "weekday"].groupby("time_block_id", observed=True)["actual_ridership"].sum()
weekend_total = rid_with_block.loc[rid_with_block["day_type"] == "weekend"].groupby("time_block_id", observed=True)["actual_ridership"].sum()

normalised_curve = pd.DataFrame({
    "weekday_share_pct": (weekday_total / weekday_total.sum() * 100).round(1),
    "weekend_share_pct": (weekend_total / weekend_total.sum() * 100).round(1),
}).reindex(dim_slot["time_block_id"])
normalised_curve

,weekday_share_pct,weekend_share_pct
time_block_id,,
1,NaN,NaN
2,20.9,9.1
3,27.0,24.1
4,12.1,31.4
5,30.1,25.7
6,9.8,9.8


**Read — this is the exit-criterion deliverable.** Weekday ridership peaks
sharply in block 5 (16:00–20:00, 30.1% of the day's riders) and block 3
(08:00–12:00, 27.0%) — the two commute peaks. Weekend ridership shifts later
and flattens: block 4 (12:00–16:00, 31.4%) is the weekend peak, not block 2 or
3 — a genuinely different curve shape, not just a scaled-down weekday one.
**A single daypart curve reused for both day types would misplace the
weekend peak by a full block.** This directly serves the plan's "briefs ask
for weekend weighting" note: the weighting has to change *which* block is
favoured, not just how much.

## Carry-forward

| Output | Consumed by |
| --- | --- |
| `scale` is not an ordinal footfall proxy (§1.6.1) | D1 static exposure model — use `est_daily_footfall` directly |
| Validated 0.3–0.5km POI-proximity radius (§1.6.2) | D1 static exposure model, Phase 4 walking-radius brief resolution |
| `side_of_road` independent of distance (§1.6.2) | D1 visibility modifier — keep as a separate term |
| Event type/attendance/radius table, weekly_season needs no expansion (§1.6.3) | Phase 6 event-surge component |
| Zone discriminant ranking, `dominant_occupation` confirmed strongest (§1.6.4) | D1 semantic audience labelling |
| Actuals-over-schedule rule; holiday ≠ weekend (§1.6.5) | Phase 6 demand baseline, D1 exposure model |
| Normalised weekday/weekend daypart curves; block 1 has no ridership signal (§1.6.6) | D1 static + mobile exposure models, Phase 4 weekend-weighting resolution |

**Next — Step 1.7 (data-quality register & cold-start census).**